# ForkWise Full Stack Provisioner

Run each cell **in order**. If a step fails, fix the issue and re-run that cell.  
All access info is printed in the final cell and saved to `SESSION_INFO.txt`.

In [ ]:
import subprocess, os, sys, time, textwrap, getpass
from datetime import datetime
from pathlib import Path
from IPython.display import display, Markdown, HTML

REPO_ROOT = Path.cwd()
# Adjust if notebook is not at repo root:
if not (REPO_ROOT / "infra").exists():
    for p in [REPO_ROOT / "ml-sys-ops-project", Path("/home/jovyan/work/ml-sys-ops-project"), Path.home() / "ml-sys-ops-project"]:
        if (p / "infra").exists():
            REPO_ROOT = p
            break

TF_DIR      = REPO_ROOT / "infra" / "tf" / "kvm"
ANSIBLE_DIR = REPO_ROOT / "infra" / "ansible"
LOG_DIR     = REPO_ROOT / "provision_logs"
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE    = LOG_DIR / f"provision_{datetime.now():%Y%m%d_%H%M%S}.log"
SESSION_FILE = REPO_ROOT / "SESSION_INFO.txt"
_log_fh = open(LOG_FILE, "a")

SESSION = []  # accumulates access info

def log(msg, level="INFO"):
    ts = datetime.now().strftime("%H:%M:%S")
    color = {"OK": "green", "WARN": "orange", "FAIL": "red", "INFO": "blue"}[level]
    display(HTML(f"<span style='color:{color}'>[{ts}] [{level}]</span> {msg}"))
    _log_fh.write(f"[{ts}] [{level}] {msg}\n"); _log_fh.flush()

def record(label, value):
    SESSION.append(f"{label}: {value}")
    log(f"→ {label}: {value}", "OK")

def run(cmd, cwd=None, env=None, check=True, timeout=600):
    log(f"<code>$ {cmd}</code>")
    merged = {**os.environ, **(env or {})}
    r = subprocess.run(cmd, shell=True, cwd=cwd, env=merged,
                       capture_output=True, text=True, timeout=timeout)
    if r.stdout.strip():
        # truncate long output
        out = r.stdout.strip()
        if len(out) > 2000:
            out = out[:2000] + "\n... (truncated)"
        print(out)
    if r.stderr.strip():
        print(r.stderr.strip()[:1000])
    _log_fh.write(r.stdout + "\n" + r.stderr + "\n"); _log_fh.flush()
    if check and r.returncode != 0:
        log(f"Command failed (rc={r.returncode}). Fix and re-run this cell.", "FAIL")
        return None
    return r

log(f"Repo root: {REPO_ROOT}", "OK")
log(f"Log file:  {LOG_FILE}", "OK")

## Step 0 — Credentials

Fill in your values below. Secrets use `getpass` so they won't be saved in notebook output.

In [ ]:
CFG = {}

CFG["SUFFIX"]           = input("Chameleon project suffix (e.g. proj01): ") or "proj01"
CFG["KEYPAIR_NAME"]     = input("Chameleon keypair name: ")
CFG["SSH_PRIVATE_KEY"]  = input("SSH private key path [/work/.ssh/id_rsa]: ") or "/work/.ssh/id_rsa"
CFG["FLOATING_IP"]      = input("node1 floating IP (blank to get from TF): ") or ""
CFG["OS_ACCESS_KEY"]    = input("OpenStack object-store access key: ")
CFG["OS_SECRET_KEY"]    = getpass.getpass("OpenStack object-store secret key: ")
CFG["GHCR_USER"]        = input("GitHub username for GHCR: ")
CFG["GHCR_TOKEN"]       = getpass.getpass("GHCR token (read:packages): ")
CFG["SERVING_IMAGE"]    = input("Serving image [ghcr.io/itsnotaka/subst-serving-onnx:demo]: ") or "ghcr.io/itsnotaka/subst-serving-onnx:demo"
CFG["AUTOMATION_IMAGE"] = input("Automation image [ghcr.io/itsnotaka/forkwise-automation:demo]: ") or "ghcr.io/itsnotaka/forkwise-automation:demo"
CFG["FLAVOR_ID"]        = input("Reserved flavor UUID (blank for m1.medium): ") or ""

log("Credentials captured (secrets hidden)", "OK")

## Step 0.5 — Verify CLI tools

In [ ]:
missing = []
for tool in ["terraform", "ansible-playbook", "kubectl", "docker", "ssh"]:
    r = subprocess.run(f"which {tool}", shell=True, capture_output=True)
    if r.returncode != 0:
        missing.append(tool)
        log(f"✗ {tool} — NOT FOUND", "FAIL")
    else:
        log(f"✓ {tool}", "OK")

if missing:
    log(f"Install missing tools before continuing: {', '.join(missing)}", "FAIL")
else:
    log("All tools present", "OK")

## Step 1 — Provision VMs (Terraform)

**Prerequisite:** `clouds.yaml` must exist at `infra/tf/kvm/clouds.yaml`.

In [ ]:
# Write terraform.tfvars
tfvars = TF_DIR / "terraform.tfvars"
tfvars.write_text(textwrap.dedent(f"""\
    suffix             = "{CFG['SUFFIX']}"
    key                = "{CFG['KEYPAIR_NAME']}"
    image_name         = "CC-Ubuntu24.04"
    flavor_id          = "{CFG['FLAVOR_ID']}"
    flavor_name        = "m1.medium"
    sharednet_name     = "sharednet1"
    floating_ip_pool   = "public"
    security_group_names = ["default"]
"""))
log(f"Wrote {tfvars}", "OK")

if not (TF_DIR / "clouds.yaml").exists():
    log("clouds.yaml NOT FOUND in infra/tf/kvm/ — copy it there first!", "FAIL")
else:
    log("clouds.yaml found", "OK")

In [ ]:
tf_env = {
    "OS_CLIENT_CONFIG_FILE": str(TF_DIR / "clouds.yaml"),
    "OS_CLOUD": "openstack",
}

run("terraform init",             cwd=TF_DIR, env=tf_env)
run("terraform validate",         cwd=TF_DIR, env=tf_env)
run("terraform plan",             cwd=TF_DIR, env=tf_env)
run("terraform apply -auto-approve", cwd=TF_DIR, env=tf_env, timeout=900)

# Capture floating IP
r = run("terraform output -raw floating_ip", cwd=TF_DIR, env=tf_env)
if r:
    CFG["FLOATING_IP"] = r.stdout.strip()
    record("Floating IP", CFG["FLOATING_IP"])
    record("Private IPs", "node1=192.168.1.11  node2=192.168.1.12  node3=192.168.1.13")

## Step 2 — Bootstrap k3s (Ansible)

In [ ]:
fip = CFG["FLOATING_IP"]
if not fip:
    log("FLOATING_IP is empty — run Step 1 first or set it manually above.", "FAIL")
else:
    # Write ansible.cfg
    cfg = ANSIBLE_DIR / "ansible.cfg"
    cfg.write_text(textwrap.dedent(f"""\
        [defaults]
        inventory = ./inventory.yml
        stdout_callback = default
        callback_result_format = yaml
        host_key_checking = False
        retry_files_enabled = False

        [ssh_connection]
        ssh_args = -o ProxyJump=cc@{fip} -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ControlMaster=auto -o ControlPersist=60s
        pipelining = True
    """))
    log(f"ansible.cfg written with floating IP {fip}", "OK")

    # ssh-agent
    run(f"eval $(ssh-agent -s) && ssh-add {CFG['SSH_PRIVATE_KEY']}", cwd=ANSIBLE_DIR, check=False)

    playbooks = [
        ("general/hello_host.yml",           "Connectivity check",  120),
        ("pre_k8s/pre_k8s_configure.yml",    "OS-level prep",       300),
        ("k8s/install_k3s.yml",              "k3s install",         600),
        ("post_k8s/post_k8s_configure.yml",  "Post-k3s config",     300),
    ]
    for pb, desc, tout in playbooks:
        log(f"Running: {desc}")
        result = run(f"ansible-playbook -i inventory.yml {pb}", cwd=ANSIBLE_DIR, timeout=tout)
        if result and result.returncode == 0:
            log(f"✓ {desc}", "OK")
        else:
            log(f"✗ {desc} — fix and re-run this cell", "FAIL")
            break

    # Verify
    log("Verifying cluster…")
    run(f"ssh -o StrictHostKeyChecking=no cc@{fip} 'kubectl get nodes -o wide'", check=False)
    record("k3s cluster", "3 nodes")

## Step 3 — GHCR Docker Login

In [ ]:
run(f"echo '{CFG['GHCR_TOKEN']}' | docker login ghcr.io -u {CFG['GHCR_USER']} --password-stdin")
log("GHCR login done", "OK")

## Step 4 — Deploy Mealie + Serving

In [ ]:
run(
    f"ansible-playbook -i inventory.yml deploy/deploy_apps.yml "
    f"-e serving_image={CFG['SERVING_IMAGE']}",
    cwd=ANSIBLE_DIR, timeout=600,
)
fip = CFG["FLOATING_IP"]
record("Mealie URL",         f"http://{fip}:30090")
record("Serving URL",        f"http://{fip}:30080")
record("Mealie SSH tunnel",  f"ssh -N -L 9000:127.0.0.1:30090 cc@{fip}")
record("Serving SSH tunnel", f"ssh -N -L 8000:127.0.0.1:30080 cc@{fip}")

## Step 5 — Create K8s Secrets

In [ ]:
fip = CFG["FLOATING_IP"]
ssh = f"ssh -o StrictHostKeyChecking=no cc@{fip}"

# os-credentials
for ns in ["monitoring-proj01", "staging-proj01", "canary-proj01", "production-proj01"]:
    run(f"{ssh} 'kubectl create namespace {ns} --dry-run=client -o yaml | kubectl apply -f - && "
        f"kubectl create secret generic os-credentials -n {ns} "
        f"--from-literal=OS_ENDPOINT=https://chi.tacc.chameleoncloud.org:7480 "
        f"--from-literal=OS_ACCESS_KEY={CFG["OS_ACCESS_KEY"]} "
        f"--from-literal=OS_SECRET_KEY={CFG["OS_SECRET_KEY"]} "
        f"--dry-run=client -o yaml | kubectl apply -f -'", check=False)

# s3-credentials
for ns in ["forkwise-data", "forkwise-serving"]:
    run(f"{ssh} 'kubectl create namespace {ns} --dry-run=client -o yaml | kubectl apply -f - && "
        f"kubectl create secret generic s3-credentials -n {ns} "
        f"--from-literal=access-key={CFG["OS_ACCESS_KEY"]} "
        f"--from-literal=secret-key={CFG["OS_SECRET_KEY"]} "
        f"--dry-run=client -o yaml | kubectl apply -f -'", check=False)

# GHCR pull secret
run(f"{ssh} 'kubectl create secret docker-registry ghcr-pull -n forkwise-data "
    f"--docker-server=ghcr.io "
    f"--docker-username={CFG["GHCR_USER"]} "
    f"--docker-password={CFG["GHCR_TOKEN"]} "
    f"--dry-run=client -o yaml | kubectl apply -f -'", check=False)

log("All secrets created/updated", "OK")

## Step 6 — Deploy Rollout Stack

In [ ]:
run(
    f"ansible-playbook -i inventory.yml deploy/deploy_rollout_stack.yml "
    f"-e serving_image={CFG['SERVING_IMAGE']} "
    f"-e automation_image={CFG['AUTOMATION_IMAGE']}",
    cwd=ANSIBLE_DIR, timeout=900,
)
fip = CFG["FLOATING_IP"]
record("Grafana SSH tunnel",    f"ssh -N -L 3000:127.0.0.1:30300 cc@{fip}")
record("Prometheus SSH tunnel", f"ssh -N -L 9090:127.0.0.1:30900 cc@{fip}")

## Step 7 — Deploy Data Workloads + Ingest

In [ ]:
fip = CFG["FLOATING_IP"]
ssh = f"ssh -o StrictHostKeyChecking=no cc@{fip}"

# Apply manifests
run(f"{ssh} 'kubectl apply -k /tmp/serving-k8s/apps/forkwise-data'", check=False)

# Ingest job
log("Running one-time ingest job (may take several minutes)…")
run(f"{ssh} 'kubectl apply -f /tmp/serving-k8s/apps/forkwise-data/job-ingest.yaml'", check=False)
run(f"{ssh} 'kubectl wait --for=condition=complete job/forkwise-ingest -n forkwise-data --timeout=1800s'",
    check=False, timeout=1860)

# Enable live workloads
log("Enabling live workloads…")
run(f"{ssh} 'kubectl scale deployment/data-generator -n forkwise-data --replicas=1'", check=False)
run(f"""{ssh} 'kubectl patch cronjob batch-pipeline -n forkwise-data -p "{{\\"spec\\":{{\\"suspend\\":false}}}}"'""", check=False)
run(f"""{ssh} 'kubectl patch cronjob drift-monitor  -n forkwise-data -p "{{\\"spec\\":{{\\"suspend\\":false}}}}"'""", check=False)
log("Data workloads enabled", "OK")

## Step 8 — Smoke Test

In [ ]:
fip = CFG["FLOATING_IP"]
ssh = f"ssh -o StrictHostKeyChecking=no cc@{fip}"

checks = [
    ("kubectl get nodes",     "Cluster nodes"),
    ("kubectl get pods -A",   "All pods"),
    ("kubectl get svc -A",    "All services"),
    ("kubectl get cronjobs -A", "All cronjobs"),
    ("curl -sf http://127.0.0.1:30080/health || echo 'serving not ready'", "Serving health"),
    ("curl -sf http://127.0.0.1:30090 -o /dev/null && echo 'mealie ok' || echo 'mealie not ready'", "Mealie health"),
]
for cmd, desc in checks:
    log(f"Check: {desc}")
    run(f"{ssh} '{cmd}'", check=False)

## Session Info — Save This!

In [ ]:
fip = CFG.get("FLOATING_IP", "UNKNOWN")

info = f"""
╔══════════════════════════════════════════════════════════════╗
║              FORKWISE SESSION INFO                         ║
║              {datetime.now():%Y-%m-%d %H:%M:%S}                              ║
╚══════════════════════════════════════════════════════════════╝

── SSH ACCESS ─────────────────────────────────────────────────
Control node:   ssh cc@{fip}
Worker node2:   ssh -J cc@{fip} cc@192.168.1.12
Worker node3:   ssh -J cc@{fip} cc@192.168.1.13

── SERVICE TUNNELS (from your laptop) ─────────────────────────
Mealie:         ssh -N -L 9000:127.0.0.1:30090 cc@{fip}
                → http://localhost:9000
Serving:        ssh -N -L 8000:127.0.0.1:30080 cc@{fip}
                → curl http://localhost:8000/health
Grafana:        ssh -N -L 3000:127.0.0.1:30300 cc@{fip}
                → http://localhost:3000
Prometheus:     ssh -N -L 9090:127.0.0.1:30900 cc@{fip}
                → http://localhost:9090
Feedback:       kubectl port-forward svc/subst-feedback -n forkwise-data 8001:8001
                → curl http://localhost:8001/health

── IMAGES ─────────────────────────────────────────────────────
Serving:        {CFG.get('SERVING_IMAGE', 'N/A')}
Automation:     {CFG.get('AUTOMATION_IMAGE', 'N/A')}
Data (GHCR):    ghcr.io/itsnotaka/forkwise-{{ingest,feedback,batch,generator}}:demo

── NAMESPACES ─────────────────────────────────────────────────
forkwise-app          Mealie UI + Postgres
forkwise-serving      substitution-serving + rollback cronjob
forkwise-data         feedback, generator, batch, drift, training
monitoring-proj01     Prometheus, Grafana, automation
staging-proj01        Staging serving
canary-proj01         Canary serving
production-proj01     Production serving

── LOG ────────────────────────────────────────────────────────
{LOG_FILE}
"""

if SESSION:
    info += "\n── RECORDED DURING PROVISION ───────────────────────────────\n"
    for line in SESSION:
        info += f"    {line}\n"

SESSION_FILE.write_text(info)
display(Markdown(f"```\n{info}\n```"))
log(f"Saved to {SESSION_FILE}", "OK")